In [14]:
%cd Proyecto_Big_Data

[Errno 2] No such file or directory: 'Proyecto_Big_Data'
/home/jovyan/work/Proyecto_Big_Data


In [15]:
import boto3
import pandas as pd
import json
import io

# Forzamos la creación de una nueva sesión para que lea el archivo del disco
session = boto3.Session()
s3 = session.client('s3')

BUCKET_NAME = "proyecto-demanda-clima"

def probar_lectura():
    try:
        # Prueba con el archivo que fallaba
        obj = s3.get_object(Bucket=BUCKET_NAME, Key="bronce/demanda/demanda_2026.json")
        print("✅ ¡CONSEGUIDO! Acceso recuperado desde el archivo de credenciales.")
        return obj
    except Exception as e:
        print(f"❌ Sigue fallando: {e}")
        return None

obj = probar_lectura()

✅ ¡CONSEGUIDO! Acceso recuperado desde el archivo de credenciales.


In [16]:
import os
import subprocess
import pandas as pd
import requests
import boto3
import io
import re
from pathlib import Path
from datetime import datetime

# --- 1. CONFIGURACIÓN DE AWS Y VALIDACIÓN ---
BUCKET_NAME = "proyecto-demanda-clima"

# Forzamos el uso de la sesión actual del laboratorio
session = boto3.Session()
s3_client = session.client('s3')

def validar_conexion():
    """Verifica si tenemos permiso de escritura en el bucket antes de empezar"""
    print("🔐 Validando permisos en S3...")
    try:
        s3_client.list_objects_v2(Bucket=BUCKET_NAME, MaxKeys=1)
        # Intento de escritura de un archivo de test
        s3_client.put_object(Bucket=BUCKET_NAME, Key="test_conexion.txt", Body="OK")
        s3_client.delete_object(Bucket=BUCKET_NAME, Key="test_conexion.txt")
        print("✅ Conexión establecida y permisos de escritura confirmados.\n")
        return True
    except Exception as e:
        print(f"❌ ERROR DE ACCESO A S3: {e}")
        print("👉 Refresca tus credenciales en ~/.aws/credentials y vuelve a intentarlo.")
        return False

def verificar_archivos_existentes(anio_inicio=2013):
    """Verifica si los archivos de clima ya existen en S3"""
    ahora = datetime.now()
    archivos_existentes = []
    archivos_faltantes = []
    
    for anio in range(anio_inicio, ahora.year + 1):
        nombre_csv = f"clima_{anio}_bronce.csv"
        try:
            s3_client.head_object(Bucket=BUCKET_NAME, Key=f"bronce/clima/{nombre_csv}")
            archivos_existentes.append(nombre_csv)
        except:
            archivos_faltantes.append(nombre_csv)
    
    return archivos_existentes, archivos_faltantes

# --- 2. CONFIGURACIÓN DE RUTAS Y LIMPIEZA ---
UNRAR_EXE = '/usr/bin/unrar' 
FOLDER_BRONCE = "./datos_clima_bronce"
FOLDER_EXTRACT = "./datos_clima_temp"
BASE_URL = "https://datosclima.es/capturadatos/"

Path(FOLDER_BRONCE).mkdir(parents=True, exist_ok=True)
Path(FOLDER_EXTRACT).mkdir(parents=True, exist_ok=True)

# --- 3. FUNCIONES DE APOYO ---

def limpiar_valor(v):
    if pd.isna(v) or v == '' or v == 'nan': 
        return None
    match = re.search(r"[-+]?\d*\.\d+|\d+", str(v))
    return float(match.group()) if match else None

def subir_a_s3_directo(df, nombre_s3):
    """Envía el DataFrame a la Capa Bronce en S3"""
    try:
        csv_buffer = io.StringIO()
        df.to_csv(csv_buffer, index=False)
        s3_client.put_object(
            Bucket=BUCKET_NAME, 
            Key=f"bronce/clima/{nombre_s3}", 
            Body=csv_buffer.getvalue()
        )
        print(f"   ✅ [S3] Persistido: bronce/clima/{nombre_s3}")
    except Exception as e:
        print(f"   ❌ [S3] FALLO EN SUBIDA: {e}")
        raise e # Detenemos el script si no puede subir para no perder tiempo

# --- 4. MOTOR PRINCIPAL ---

def ejecutar_pipeline_clima(anio_inicio=2013):
    if not validar_conexion(): 
        return
    
    # VERIFICACIÓN INICIAL: Comprobar si los archivos ya existen
    print("📋 Verificando archivos existentes en S3...")
    existentes, faltantes = verificar_archivos_existentes(anio_inicio)
    
    if existentes:
        print(f"\n✅ Archivos ya procesados ({len(existentes)}):")
        for arch in existentes:
            print(f"   ✓ {arch}")
    
    if not faltantes:
        print("\n🎉 ¡Todos los archivos están disponibles en S3!")
        print("⏭️  Saltando descarga. Continúa con las siguientes celdas.")
        return
    
    print(f"\n📥 Faltantes: {len(faltantes)} archivo(s) por procesar")
    for arch in faltantes:
        print(f"   → {arch}")
    print("\n🚀 Iniciando descarga e ingesta...\n")
    
    ahora = datetime.now()
    
    for anio in range(anio_inicio, ahora.year + 1):
        nombre_csv_anual = f"clima_{anio}_bronce.csv"
        
        # SKIP: Si el archivo ya existe, no lo vuelvas a procesar
        if nombre_csv_anual in existentes:
            print(f"⏭️  AÑO {anio}: Ya existe en S3. Omitiendo...")
            continue
        
        print(f"\n📅 PROCESANDO AÑO: {anio}")
        
        lista_anual = []
        mes_inicio = 5 if anio == 2013 else 1
        
        for mes in range(mes_inicio, 13):
            if anio == ahora.year and mes > ahora.month: break
                
            nombre_rar = f"Aemet{anio}-{mes:02d}.rar"
            url = f"{BASE_URL}{nombre_rar}"
            path_rar = os.path.join(FOLDER_BRONCE, nombre_rar)
            
            try:
                res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=20)
                if res.status_code == 200:
                    with open(path_rar, "wb") as f:
                        f.write(res.content)
                    
                    subprocess.run([UNRAR_EXE, 'e', '-o+', path_rar, FOLDER_EXTRACT], capture_output=True)
                    ficheros = sorted([f for f in os.listdir(FOLDER_EXTRACT) if f.endswith(('.xls', '.xlsx'))])
                    
                    for i, f in enumerate(ficheros):
                        path_f = os.path.join(FOLDER_EXTRACT, f)
                        fecha_registro = f"{anio}-{str(mes).zfill(2)}-{str(i+1).zfill(2)}"
                        
                        try:
                            df_preview = pd.read_excel(path_f, header=None, nrows=10).astype(str)
                            idx_header = 4
                            for fila_idx, row in df_preview.iterrows():
                                if "estación" in " ".join(row.values).lower():
                                    idx_header = fila_idx
                                    break
                            
                            df_dia = pd.read_excel(path_f, skiprows=idx_header)
                            df_dia.columns = [str(c).replace('\n', ' ').strip() for c in df_dia.columns]
                            df_dia = df_dia[df_dia['Estación'].notna()].copy()
                            df_dia['fecha'] = fecha_registro
                            
                            for col in ['Temperatura máxima (ºC)', 'Temperatura mínima (ºC)', 'Temperatura media (ºC)', 'Precipitación (mm)']:
                                if col in df_dia.columns:
                                    df_dia[col] = df_dia[col].apply(limpiar_valor)
                            
                            lista_anual.append(df_dia)
                        except: continue
                        if os.path.exists(path_f): os.remove(path_f)
                    
                    if os.path.exists(path_rar): os.remove(path_rar)
                    print(f"   ✔️ Mes {mes:02d} procesado localmente.")
            except Exception as e:
                print(f"   ❌ Error mes {mes:02d}: {e}")

        # Consolidación anual y subida
        if lista_anual:
            df_anio = pd.concat(lista_anual, ignore_index=True)
            df_anio = df_anio.loc[:, ~df_anio.columns.str.contains('^Unnamed')]
            cols_finales = ['fecha'] + [c for c in df_anio.columns if c != 'fecha']
            
            # Aquí es donde realmente se guarda en S3
            subir_a_s3_directo(df_anio[cols_finales], nombre_csv_anual)

# LANZAR
ejecutar_pipeline_clima(anio_inicio=2013)

🔐 Validando permisos en S3...
✅ Conexión establecida y permisos de escritura confirmados.

📋 Verificando archivos existentes en S3...

✅ Archivos ya procesados (14):
   ✓ clima_2013_bronce.csv
   ✓ clima_2014_bronce.csv
   ✓ clima_2015_bronce.csv
   ✓ clima_2016_bronce.csv
   ✓ clima_2017_bronce.csv
   ✓ clima_2018_bronce.csv
   ✓ clima_2019_bronce.csv
   ✓ clima_2020_bronce.csv
   ✓ clima_2021_bronce.csv
   ✓ clima_2022_bronce.csv
   ✓ clima_2023_bronce.csv
   ✓ clima_2024_bronce.csv
   ✓ clima_2025_bronce.csv
   ✓ clima_2026_bronce.csv

🎉 ¡Todos los archivos están disponibles en S3!
⏭️  Saltando descarga. Continúa con las siguientes celdas.


In [30]:
# 1. Aseguramos que la sesión sea la que te dio éxito
session = boto3.Session()
s3_exito = session.client('s3')

# 2. Procesar todos los JSON de demanda en S3
def procesar_demanda_plata(cliente_s3):
    print("📥 Iniciando proceso de aplanado de demanda (todos los JSON)...")

    try:
        # Listar todos los archivos JSON de demanda
        paginator = cliente_s3.get_paginator('list_objects_v2')
        pages = paginator.paginate(Bucket=BUCKET_NAME, Prefix="bronce/demanda/")

        claves_json = []
        for page in pages:
            for obj in page.get('Contents', []):
                key = obj['Key']
                if key.endswith('.json'):
                    claves_json.append(key)

        if not claves_json:
            print("❌ No se encontraron JSON de demanda en bronce/demanda/")
            return None

        print(f"📂 JSON detectados: {len(claves_json)}")

        lista_df = []
        for key in sorted(claves_json):
            try:
                obj = cliente_s3.get_object(Bucket=BUCKET_NAME, Key=key)
                data = json.loads(obj['Body'].read().decode('utf-8'))

                valores = data['included'][0]['attributes']['values']
                df_temp = pd.DataFrame(valores)[['datetime', 'value']].copy()
                df_temp.columns = ['fecha', 'demanda_mw']
                lista_df.append(df_temp)
                print(f"   ✓ {key}")
            except Exception as e:
                print(f"   ⚠️ Error en {key}: {e}")

        if not lista_df:
            print("❌ No se pudo procesar ningún JSON de demanda")
            return None

        # Unificar, limpiar y ordenar
        df = pd.concat(lista_df, ignore_index=True)
        df['fecha'] = pd.to_datetime(df['fecha'], utc=True, errors='coerce').dt.tz_localize(None)
        df['demanda_mw'] = pd.to_numeric(df['demanda_mw'], errors='coerce')

        # Eliminar duplicados por fecha (si hay solapes entre ficheros)
        df = df.dropna(subset=['fecha', 'demanda_mw'])
        df = df.sort_values('fecha').drop_duplicates(subset=['fecha'], keep='last').reset_index(drop=True)

        # Persistir
        df.to_parquet("demanda_plata.parquet", engine='pyarrow', index=False)

        print(f"✅ Demanda plata generada correctamente con {len(df)} filas")
        print(f"📅 Rango: {df['fecha'].min().date()} -> {df['fecha'].max().date()}")
        return df

    except Exception as e:
        print(f"❌ Error al procesar demanda: {e}")
        return None


# Ejecutar
df_demanda_plata = procesar_demanda_plata(s3_exito)

📥 Iniciando proceso de aplanado de demanda (todos los JSON)...
📂 JSON detectados: 1
   ✓ bronce/demanda/demanda_2026.json
✅ Demanda plata generada correctamente con 118 filas
📅 Rango: 2025-12-31 -> 2026-04-27


In [18]:
import holidays
import pandas as pd
import os

# 1. Generar los datos
fecha_inicio = '2013-05-01'
fecha_fin = '2026-12-31'
fechas = pd.date_range(start=fecha_inicio, end=fecha_fin)
df_festivos = pd.DataFrame({'fecha': fechas})

# Festivos de España (Castilla y León)
festivos_esp = holidays.Spain(years=range(2013, 2027), subdiv='CL')
df_festivos['nombre_festivo'] = df_festivos['fecha'].map(festivos_esp)
df_festivos['es_festivo'] = df_festivos['nombre_festivo'].notna().astype(int)

# 2. Guardar CSV temporal
nombre_archivo = "festivos_espana_bronce.csv"
df_festivos.to_csv(nombre_archivo, index=False)

# 3. Subir a S3 usando el cliente que ya sabemos que funciona (s3_exito)
try:
    s3_exito.upload_file(nombre_archivo, BUCKET_NAME, f"bronce/festivos/{nombre_archivo}")
    print(f"✅ Archivo subido correctamente a s3://{BUCKET_NAME}/bronce/festivos/{nombre_archivo}")
    
    # Limpiamos el archivo local
    os.remove(nombre_archivo)
except Exception as e:
    print(f"❌ Error al subir: {e}")

✅ Archivo subido correctamente a s3://proyecto-demanda-clima/bronce/festivos/festivos_espana_bronce.csv


In [19]:
import pandas as pd
import io

def crear_parquet_festivos_faltante(cliente_s3):
    print("📥 Recuperando festivos de S3 para crear el Parquet...")
    try:
        # 1. Descargar el CSV de la Capa Bronce
        key_s3 = "bronce/festivos/festivos_espana_bronce.csv"
        obj = cliente_s3.get_object(Bucket=BUCKET_NAME, Key=key_s3)
        df_festivos = pd.read_csv(io.BytesIO(obj['Body'].read()))
        
        # 2. Asegurar formato de fecha
        df_festivos['fecha'] = pd.to_datetime(df_festivos['fecha'])
        
        # 3. Guardar como Parquet localmente
        df_festivos.to_parquet("festivos_plata.parquet", engine='pyarrow')
        print("✅ Archivo 'festivos_plata.parquet' creado exitosamente.")
        
    except Exception as e:
        print(f"❌ No se pudo crear el archivo: {e}")

# Ejecuta esto antes de volver a intentar la Gran Unión
crear_parquet_festivos_faltante(s3_exito)

📥 Recuperando festivos de S3 para crear el Parquet...
✅ Archivo 'festivos_plata.parquet' creado exitosamente.


In [20]:
import pandas as pd
import io

def fabricar_clima_plata(cliente_s3):
    print("🛠️ Fabricando clima_plata.parquet a partir de S3...")
    
    # 1. Listar los CSV de clima en S3
    paginator = cliente_s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=BUCKET_NAME, Prefix="bronce/clima/")
    
    lista_dfs = []
    for page in pages:
        for obj in page.get('Contents', []):
            # Solo leemos si es CSV y es realmente de clima (evitamos el de festivos)
            if obj['Key'].endswith('.csv') and 'clima' in obj['Key'].lower():
                print(f"  -> Procesando: {obj['Key']}")
                r = cliente_s3.get_object(Bucket=BUCKET_NAME, Key=obj['Key'])
                df_temp = pd.read_csv(io.BytesIO(r['Body'].read()))
                lista_dfs.append(df_temp)
    
    if lista_dfs:
        # 2. Unir todos los archivos (2013-2026) en uno solo
        df_total = pd.concat(lista_dfs, ignore_index=True)
        
        # 3. Limpieza de nombres de columnas (esto es lo que lo convierte en "Capa Plata")
        df_total.columns = [c.replace('\n', ' ').strip() for c in df_total.columns]
        
        # 4. PERSISTENCIA: Aquí es donde se crea el archivo en tu local
        df_total.to_parquet("clima_plata.parquet", engine='pyarrow')
        
        print("✅ Archivo 'clima_plata.parquet' generado en tu local.")
        print(f"📊 Total de registros procesados: {len(df_total)}")
    else:
        print("❌ No se encontraron archivos válidos en S3 para procesar.")

# Ejecuta esta función
fabricar_clima_plata(s3_exito)

🛠️ Fabricando clima_plata.parquet a partir de S3...
  -> Procesando: bronce/clima/clima_2013_bronce.csv
  -> Procesando: bronce/clima/clima_2014_bronce.csv
  -> Procesando: bronce/clima/clima_2015_bronce.csv
  -> Procesando: bronce/clima/clima_2016_bronce.csv
  -> Procesando: bronce/clima/clima_2017_bronce.csv
  -> Procesando: bronce/clima/clima_2018_bronce.csv
  -> Procesando: bronce/clima/clima_2019_bronce.csv
  -> Procesando: bronce/clima/clima_2020_bronce.csv
  -> Procesando: bronce/clima/clima_2021_bronce.csv
  -> Procesando: bronce/clima/clima_2022_bronce.csv
  -> Procesando: bronce/clima/clima_2023_bronce.csv
  -> Procesando: bronce/clima/clima_2024_bronce.csv
  -> Procesando: bronce/clima/clima_2025_bronce.csv
  -> Procesando: bronce/clima/clima_2026_bronce.csv
✅ Archivo 'clima_plata.parquet' generado en tu local.
📊 Total de registros procesados: 3753816


In [21]:
def reconstruir_clima_plata_seguro(cliente_s3):
    print("🌩️ Reconstruyendo clima_plata.parquet de forma segura...")
    paginator = cliente_s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=BUCKET_NAME, Prefix="bronce/clima/")
    
    lista_dfs = []
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                # FILTRO CRÍTICO: Debe ser CSV y contener la palabra clima
                if obj['Key'].endswith('.csv') and 'clima' in obj['Key'].lower():
                    print(f"  -> Procesando archivo válido: {obj['Key']}")
                    resp = cliente_s3.get_object(Bucket=BUCKET_NAME, Key=obj['Key'])
                    df_temp = pd.read_csv(io.BytesIO(resp['Body'].read()))
                    lista_dfs.append(df_temp)
    
    if lista_dfs:
        df_clima_total = pd.concat(lista_dfs, ignore_index=True)
        # Limpieza de cabeceras
        df_clima_total.columns = [c.replace('\n', ' ').strip() for c in df_clima_total.columns]
        # Guardar
        df_clima_total.to_parquet("clima_plata.parquet", engine='pyarrow')
        print(f"✅ ¡ÉXITO! Clima reconstruido con {len(df_clima_total)} filas.")
        print(f"📋 Columnas finales: {df_clima_total.columns.tolist()}")
        return True
    else:
        print("❌ ERROR: No se han encontrado archivos que contengan 'clima' en la ruta S3.")
        return False

reconstruir_clima_plata_seguro(s3_exito)

🌩️ Reconstruyendo clima_plata.parquet de forma segura...
  -> Procesando archivo válido: bronce/clima/clima_2013_bronce.csv
  -> Procesando archivo válido: bronce/clima/clima_2014_bronce.csv
  -> Procesando archivo válido: bronce/clima/clima_2015_bronce.csv
  -> Procesando archivo válido: bronce/clima/clima_2016_bronce.csv
  -> Procesando archivo válido: bronce/clima/clima_2017_bronce.csv
  -> Procesando archivo válido: bronce/clima/clima_2018_bronce.csv
  -> Procesando archivo válido: bronce/clima/clima_2019_bronce.csv
  -> Procesando archivo válido: bronce/clima/clima_2020_bronce.csv
  -> Procesando archivo válido: bronce/clima/clima_2021_bronce.csv
  -> Procesando archivo válido: bronce/clima/clima_2022_bronce.csv
  -> Procesando archivo válido: bronce/clima/clima_2023_bronce.csv
  -> Procesando archivo válido: bronce/clima/clima_2024_bronce.csv
  -> Procesando archivo válido: bronce/clima/clima_2025_bronce.csv
  -> Procesando archivo válido: bronce/clima/clima_2026_bronce.csv
✅ ¡ÉX

True

In [22]:
# Vamos a leer solo el primer archivo de clima para ver sus tripas
def inspeccionar_columnas_s3(cliente_s3):
    response = cliente_s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix="bronce/clima/")
    if 'Contents' in response:
        first_key = [obj['Key'] for obj in response['Contents'] if obj['Key'].endswith('.csv')][0]
        print(f"🔍 Inspeccionando archivo: {first_key}")
        obj = cliente_s3.get_object(Bucket=BUCKET_NAME, Key=first_key)
        df_debug = pd.read_csv(io.BytesIO(obj['Body'].read()), nrows=5)
        print("📋 Columnas reales en el CSV:")
        print(df_debug.columns.tolist())
        return df_debug.columns.tolist()
    else:
        print("❌ No hay archivos en bronce/clima/")

cols_reales = inspeccionar_columnas_s3(s3_exito)

🔍 Inspeccionando archivo: bronce/clima/clima_2013_bronce.csv
📋 Columnas reales en el CSV:
['fecha', 'Estación', 'Provincia', 'Temperatura máxima (ºC)', 'Temperatura mínima (ºC)', 'Temperatura media (ºC)', 'Racha (km/h)', 'Velocidad máxima (km/h)', 'Precipitación 00-24h (mm)', 'Precipitación 00-06h (mm)', 'Precipitación 06-12h (mm)', 'Precipitación 12-18h (mm)', 'Precipitación 18-24h (mm)']


In [31]:
def generar_capa_oro_segura(cliente_s3):
    print("📀 Generando Dataset Maestro...")

    # 1. Cargar clima desde S3
    paginator = cliente_s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=BUCKET_NAME, Prefix="bronce/clima/")
    lista_clima = []

    for page in pages:
        for obj in page.get('Contents', []):
            if 'clima' in obj['Key'].lower() and obj['Key'].endswith('.csv'):
                r = cliente_s3.get_object(Bucket=BUCKET_NAME, Key=obj['Key'])
                df_temp = pd.read_csv(io.BytesIO(r['Body'].read()))
                lista_clima.append(df_temp)

    if not lista_clima:
        print("❌ Error crítico: No hay datos de clima válidos.")
        return None

    df_clima = pd.concat(lista_clima, ignore_index=True)
    df_clima.columns = [str(c).replace('\n', ' ').strip() for c in df_clima.columns]

    # 2. Cargar datasets locales
    df_demanda = pd.read_parquet("demanda_plata.parquet")
    df_festivos = pd.read_parquet("festivos_plata.parquet")

    # 3. Normalizar fechas
    print("🔧 Limpiando y normalizando fechas...")
    df_demanda['fecha'] = pd.to_datetime(df_demanda['fecha'], errors='coerce').dt.date
    df_festivos['fecha'] = pd.to_datetime(df_festivos['fecha'], errors='coerce').dt.date
    df_clima['fecha'] = pd.to_datetime(df_clima['fecha'], format='mixed', errors='coerce').dt.date

    # Eliminar filas sin fecha válida
    n_invalidas = df_clima['fecha'].isna().sum()
    if n_invalidas > 0:
        print(f"⚠️ Se eliminaron {n_invalidas} filas de clima con fecha inválida")
        df_clima = df_clima.dropna(subset=['fecha'])

    # 4. Detectar columnas meteorológicas
    c_tmax = None
    c_tmin = None
    c_tmed = None
    c_prec = None

    for c in df_clima.columns:
        c_lower = c.lower()
        if c_tmax is None and ('maxima' in c_lower or 'máxima' in c_lower or 'tmax' in c_lower or 'temp_max' in c_lower):
            c_tmax = c
        if c_tmin is None and ('minima' in c_lower or 'mínima' in c_lower or 'tmin' in c_lower or 'temp_min' in c_lower):
            c_tmin = c
        if c_tmed is None and ('media' in c_lower or 'tmed' in c_lower or 'temp_med' in c_lower):
            c_tmed = c
        if c_prec is None and ('precipit' in c_lower or c_lower in ('prec', 'lluvia', 'rain')):
            c_prec = c

    if c_tmax is None and c_tmin is None and c_tmed is None:
        print(f"❌ No se detectaron columnas de temperatura. Columnas disponibles: {df_clima.columns.tolist()}")
        return None

    # 5. Convertir columnas encontradas a numéricas
    columnas_meteo = [c for c in [c_tmax, c_tmin, c_tmed, c_prec] if c is not None]
    for c in columnas_meteo:
        df_clima[c] = pd.to_numeric(df_clima[c], errors='coerce')

    # 6. Agregación diaria de clima
    print("📊 Agregando datos meteorológicos por fecha...")
    agg_cols = [c for c in [c_tmax, c_tmin, c_tmed, c_prec] if c is not None]
    df_cl_diario = df_clima.groupby('fecha')[agg_cols].mean().reset_index()

    rename_map = {}
    if c_tmax is not None:
        rename_map[c_tmax] = 'temp_max'
    if c_tmin is not None:
        rename_map[c_tmin] = 'temp_min'
    if c_tmed is not None:
        rename_map[c_tmed] = 'temp_med'
    if c_prec is not None:
        rename_map[c_prec] = 'precipitacion'
    df_cl_diario = df_cl_diario.rename(columns=rename_map)

    # Si no viene temp_med, la calculamos
    if 'temp_med' not in df_cl_diario.columns and {'temp_max', 'temp_min'}.issubset(df_cl_diario.columns):
        df_cl_diario['temp_med'] = (df_cl_diario['temp_max'] + df_cl_diario['temp_min']) / 2

    # Si no viene precipitación, se deja en 0 para mantener esquema estable
    if 'precipitacion' not in df_cl_diario.columns:
        df_cl_diario['precipitacion'] = 0.0

    # 7. Merge final
    print("🔗 Unificando datasets...")
    master = pd.merge(df_demanda, df_cl_diario, on='fecha', how='inner')
    master = pd.merge(master, df_festivos[['fecha', 'es_festivo']], on='fecha', how='left')
    master['es_festivo'] = master['es_festivo'].fillna(0).astype(int)

    # Orden de columnas recomendado
    columnas_orden = [
        c for c in ['fecha', 'demanda_mw', 'temp_max', 'temp_min', 'temp_med', 'precipitacion', 'es_festivo']
        if c in master.columns
    ]
    master = master[columnas_orden]

    master.to_parquet("dataset_final_oro.parquet", index=False)
    print(f"🚀 ¡Dataset Oro creado con éxito! Filas: {len(master)}")
    print(f"📋 Columnas finales: {master.columns.tolist()}")
    return master


df_oro = generar_capa_oro_segura(s3_exito)

📀 Generando Dataset Maestro...
🔧 Limpiando y normalizando fechas...
⚠️ Se eliminaron 236393 filas de clima con fecha inválida
📊 Agregando datos meteorológicos por fecha...
🔗 Unificando datasets...
🚀 ¡Dataset Oro creado con éxito! Filas: 79
📋 Columnas finales: ['fecha', 'demanda_mw', 'temp_max', 'temp_min', 'temp_med', 'precipitacion', 'es_festivo']


In [26]:
%pip install plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 13.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.9/451.9 kB 11.9 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [32]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# 1. Cargar el dataset final (Capa Oro)
df = pd.read_parquet("dataset_final_oro.parquet")
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha')

# Garantizar columnas esperadas
if 'temp_med' not in df.columns and {'temp_max', 'temp_min'}.issubset(df.columns):
    df['temp_med'] = (df['temp_max'] + df['temp_min']) / 2
if 'precipitacion' not in df.columns:
    df['precipitacion'] = 0.0
if 'es_festivo' not in df.columns:
    df['es_festivo'] = 0

# 2. Crear Dashboard con subplots
fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=("Consumo Eléctrico (MW)", "Temperatura Media (°C)", "Precipitaciones (mm)"),
    row_heights=[0.5, 0.25, 0.25]
)

# --- Gráfica de Demanda ---
fig.add_trace(
    go.Scatter(x=df['fecha'], y=df['demanda_mw'], name="Demanda", line=dict(color='royalblue')),
    row=1,
    col=1
)

# Añadir festivos como puntos rojos sobre la demanda
festivos = df[df['es_festivo'] == 1]
fig.add_trace(
    go.Scatter(
        x=festivos['fecha'],
        y=festivos['demanda_mw'],
        mode='markers',
        name="Día Festivo",
        marker=dict(color='red', size=8)
    ),
    row=1,
    col=1
)

# --- Gráfica de Temperatura ---
fig.add_trace(
    go.Scatter(x=df['fecha'], y=df['temp_med'], name="Temp. Media", line=dict(color='orange')),
    row=2,
    col=1
)

# --- Gráfica de Lluvia ---
fig.add_trace(
    go.Bar(x=df['fecha'], y=df['precipitacion'], name="Lluvia", marker_color='lightblue'),
    row=3,
    col=1
)

# Estética y Layout
fig.update_layout(
    title_text="📊 DASHBOARD DE EXPLOTACIÓN: Demanda Energética y Clima",
    height=900,
    showlegend=True,
    template="plotly_white",
    hovermode="x unified"
)

fig.show()